In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

from Document import Document
from Corpus import Corpus
from SearchEngine import SearchEngine

df = pd.read_csv("corpus.csv")
corpus = Corpus("Corpus TD8")

for i, row in df.iterrows():
    auteur = row["auteur"]
    texte = row["texte"]

    phrases = texte.split(".")
    for j, p in enumerate(phrases):
        p = p.strip()
        if len(p) > 0:
            doc = Document(
                titre=f"Phrase {i}-{j}",
                auteur=auteur,
                date="2024-01-01",   # date par défaut (format OK)
                url="",              # pas d’URL
                texte=p
            )
            corpus.ajouter_document(doc)

print("Corpus construit avec :", corpus.ndoc, "documents")


moteur = SearchEngine(corpus)
print("Moteur de recherche initialisé.")

liste_auteurs = sorted(set(doc.auteur for doc in corpus.id2doc.values()))
liste_auteurs.insert(0, "Tous")

dropdown_auteur = widgets.Dropdown(
    options=liste_auteurs,
    description="Auteur : "
)

titre = widgets.Label("🔍 Interface de recherche – TD8")

champ_requete = widgets.Text(
    description="Mots-clés : ",
    placeholder="Tapez un mot…"
)

slider_nb = widgets.IntSlider(
    value=5, min=1, max=50, step=1,
    description="Résultats :"
)

bouton = widgets.Button(
    description="Rechercher",
    button_style="primary"
)

zone_sortie = widgets.Output()

# ============================================================
# 4. ACTION DU BOUTON
# ============================================================

def clique_bouton(b):
    with zone_sortie:
        clear_output()

        requete = champ_requete.value.strip().lower()
        k = slider_nb.value
        filtre = dropdown_auteur.value

        if requete == "":
            print("⚠️ Entrez un mot-clé.")
            return

        resultats = moteur.search(requete, k=1000)

        if filtre != "Tous":
            resultats = [doc for doc in resultats if doc.auteur == filtre]

        resultats = resultats[:k]

        print(f"Résultats pour '{requete}'  | Auteur = {filtre}")
        print("-" * 50)

        if len(resultats) == 0:
            print("❌ Aucun résultat trouvé.")
            return

        for doc in resultats:
            print("Auteur :", doc.auteur)
            print("Phrase :", doc.texte)
            print("-" * 50)

bouton.on_click(clique_bouton)

ui = widgets.VBox([
    titre,
    widgets.HBox([champ_requete, slider_nb]),
    dropdown_auteur,
    bouton,
    zone_sortie
])

display(ui)


Corpus construit avec : 3 documents
Moteur de recherche initialisé.
